In [ ]:
!nvidia-smi

In [ ]:
!pip install transformers
!pip install datasets
!pip install peft
!pip install accelerate
!pip install bitsandbytes

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from datasets import load_dataset

dataset = load_dataset("rootsautomation/RICO-Screen2Words")
dataset

In [ ]:
dataset["train"][0]

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")

model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)

In [ ]:
from peft import LoraConfig, get_peft_model

config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["query", "value"],
    lora_dropout=0.05,
)

model = get_peft_model(model, config)

In [ ]:
def preprocess(example):
    caption = example["captions"][0]   # take the first caption

    inputs = processor(
        images=example["image"],
        text=caption,
        return_tensors="pt"
    )

    inputs["labels"] = inputs["input_ids"]
    return inputs

In [ ]:
def preprocess(example):
    caption = example["captions"][0]

    inputs = processor(
        images=example["image"],
        text=caption,
        padding="max_length",
        truncation=True,
        max_length=50,
        return_tensors="pt"
    )

    inputs["labels"] = inputs["input_ids"]

    return {k: v.squeeze() for k, v in inputs.items()}


dataset = dataset.select(range(500))

dataset = dataset.map(preprocess)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    num_train_epochs=1,
    learning_rate=2e-5,
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

In [ ]:
trainer.train()

In [ ]:
model.push_to_hub("rico-blip-lora-model")
processor.push_to_hub("rico-blip-lora-model")

In [ ]:
from PIL import Image
import requests

url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/cats.png"

image = Image.open(requests.get(url, stream=True).raw)

inputs = processor(images=image, return_tensors="pt").to("cuda")

out = model.generate(**inputs)

processor.decode(out[0], skip_special_tokens=True)